## Integer Unpacking benchmark results

The following interaction explores the results of the benchmarks on various architectures.
The controls let you set with the following options.

#### Functions:
The different implementations being benchmark.
This is the combination of the micro architecture and the specific code.
The micro architecture such are `Neon` on `Arm64` and `SSE4.2`, `AVX2` and `AVX512` on `x86-64`.
The different code functions are:
 - **ScalarExact**: The naive function that can unpack exactly the desired number of values and that is used to wrap other fuctions that can only extract a batch of values at the time. 
 - **ScalarBatch**: A version without SIMD code that extract values in batch.
 - **Old**: The SIMD version that was previously in Apache Arrow.
 - **New**: The new SIMD implementation introduce with the work presented here.
 - **NewNoDispatch**: The same SIMD implementation, with all working types fixed to the one described.
 - **LitteIntPacker**: The work by Daniel Lemire available in the [LittleIntPacker library](https://github.com/fast-pack/LittleIntPacker) (under horizontal packing). This was added to this build for benchmark and wraped with the same bit width dispatch mechanism for a fair comparison. This is only available for the `SSE` family on `x86-64` and for `uint32_t`, but we added a few more unpacking type by extracting in a local buffer and looping on it with `static_cast` in the same way we do used for casting between different integer types.
 - **Dynamic**: Should be the best function for the architecture, wrapped in a dynamic dispatch to the apporpriate function depending on the micro architecture detected at runtime.

*Note*: The scalar functions are compiled in different files with different for different micro architectures to evaluate the capacity of the compiler to auto-vectorize the code compared the manual vectoization we do with the [xsimd library](https://github.com/xtensor-stack/xsimd/). 
Although `Neon` is always available on `Arm64`, we specifically built some files without it to isolate what the compiler auto vectorization can do.
This setting is intended for research and does not make sense in general.

#### Unpacked type:
This is the type the that the unpack function targets.

#### Packed width:
This is the the number of bits used to encode individual values.
Larger bit width can only be extracted to larger integer types.

### Full results

In [ ]:
import pathlib
import re

import polars as pl


def count_header_lines(path: pathlib.Path) -> int:
    """Return the number of lines to skip before the CSV header row.

    Reads the file lazily, looking for the first line that matches a
    CSV-style header pattern: comma-separated words (e.g. "name,iterations,real_time").
    Returns the 0-based line index, i.e. the number of preceding lines to skip.
    """
    header_re = re.compile(r"^(\w+,)+\w+\s*$")
    with path.open() as f:
        for i, line in enumerate(f):
            if header_re.match(line):
                return i
    raise ValueError(f"No CSV header found in {path}")


def clean_benchmark(df: pl.LazyFrame) -> pl.LazyFrame:
    return (
        df
        # Remove pre-aggregated data generating heterogenous rows
        .filter(~pl.col("name").str.contains(r"_(mean|median|stddev|cv)$"))
        # Remove skipped benchmarks
        .filter(~pl.col("cpu_time").is_null())
        # Unused / all nulls
        .drop(
            "label",
            "error_occurred",
            "error_message",
            "bytes_per_second",
            "items_per_second",
        )
        # Parse benchmark name into idividual components
        .with_columns(
            pl.col("name")
            .str.extract_groups(
                r"BM_Unpack(?P<unpacked_type>\w+)/"
                r"(?P<arch>\w+)-(?P<func>\w+)/"
                r"(?P<packed_width>\d+)/(?P<num_values>\d+)"
            )
            .struct.unnest()
        )
        # Extract unpacked size from width, assuming 8 bits bool
        .with_columns(
            pl.when(pl.col("unpacked_type").str.to_lowercase() == "bool")
            .then(pl.lit("Uint8"))
            .otherwise(pl.col("unpacked_type"))
            .str.to_lowercase()
            .str.strip_prefix("uint")
            .cast(pl.Int32)
            .alias("unpacked_width"),
        )
        # Cast parsed integer components
        .with_columns(
            pl.col("num_values").cast(pl.Int32), pl.col("packed_width").cast(pl.Int32)
        )
        # Remove mangled name column
        .drop("name")
        # Cast remaining strings to categorical
        .with_columns(pl.col(pl.String).cast(pl.Categorical))
    )


def load_cleaned_benchmarks(
    arch: str = "linux-64", directory: pathlib.Path | str = "data"
) -> pl.DataFrame:
    raw_path = pathlib.Path(directory) / f"benchmark-{arch}.csv"
    processed_path = pathlib.Path(directory) / f"benchmark-{arch}.parquet"

    if processed_path.exists():
        return pl.read_parquet(processed_path)

    # Variable number of rows to skip at the top
    n_comment_lines = count_header_lines(raw_path)
    lf_raw = pl.scan_csv(raw_path, skip_rows=n_comment_lines)
    df = clean_benchmark(lf_raw).collect()
    df.write_parquet(processed_path)
    return df

In [ ]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
import traitlets


class MultiSelect(widgets.VBox):
    """Generic multi-select checkbox widget with .value and observe support."""

    value = traitlets.Tuple(help="Currently selected values")

    def __init__(self, options, default_filter=None, description="", **kwargs):
        """
        Parameters
        ----------
        options : iterable
            Available choices.
        default_filter : callable, optional
            fn(option) -> bool to set initial checked state. All checked if None.
        description : str, optional
            Label shown above the checkboxes.
        """
        self._checkboxes = {}
        for opt in options:
            checked = default_filter(opt) if default_filter else True
            cb = widgets.Checkbox(value=checked, description=str(opt), indent=False)
            cb._select_key = opt
            cb.observe(self._on_change, names="value")
            self._checkboxes[opt] = cb

        children = []
        if description:
            children.append(widgets.Label(description))
        children.append(
            widgets.HBox(
                list(self._checkboxes.values()),
                layout=widgets.Layout(flex_flow="row wrap"),
            )
        )

        super().__init__(children=children, **kwargs)
        self._sync_value()

    def _on_change(self, change):
        self._sync_value()

    def _sync_value(self):
        self.value = tuple(key for key, cb in self._checkboxes.items() if cb.value)


class GroupedMultiSelect(widgets.VBox):
    """Multi-select checkboxes grouped by category with .value and observe support.

    Parameters
    ----------
    groups : dict[str, list[str]]
        Mapping of category name to list of options.
    default_filter : callable, optional
        fn(category, option) -> bool for initial checked state. All checked if None.
    """

    value = traitlets.Dict(
        help="Currently selected values per group: {category: (selected, ...)}"
    )

    def __init__(self, groups, default_filter=None, **kwargs):
        self._groups = {}  # category -> {option: Checkbox}
        children = []

        for category, options in groups.items():
            checkboxes = {}
            for opt in options:
                checked = default_filter(category, opt) if default_filter else True
                cb = widgets.Checkbox(
                    value=checked,
                    description=str(opt),
                    indent=False,
                    layout=widgets.Layout(
                        width="auto", min_width="0", margin="0 0 0 12px"
                    ),
                )
                cb.observe(self._on_change, names="value")
                checkboxes[opt] = cb
            self._groups[category] = checkboxes

            children.append(widgets.Label(f"{category}:"))
            children.append(
                widgets.HBox(
                    list(checkboxes.values()),
                    layout=widgets.Layout(flex_flow="row wrap"),
                )
            )

        super().__init__(children=children, **kwargs)
        self._sync_value()

    def _on_change(self, change):
        self._sync_value()

    def _sync_value(self):
        self.value = {
            cat: tuple(opt for opt, cb in cbs.items() if cb.value)
            for cat, cbs in self._groups.items()
        }


def build_type_sizes(df: pl.DataFrame) -> dict[str, int]:
    cols = ["unpacked_width", "unpacked_type"]
    unpacked_df = df[cols].unique().sort(by=cols)
    unpacked_dict = {
        r["unpacked_type"]: r["unpacked_width"]
        for r in unpacked_df.iter_rows(named=True)
    }
    unpacked_dict["Bool"] = 1
    return unpacked_dict


def make_unpacked_type_wt(df: pl.DataFrame) -> MultiSelect:
    unpacked_dict = build_type_sizes(df)
    return MultiSelect(
        options=list(unpacked_dict.keys()),
        default_filter=lambda v: "32" in v,
    )


def make_arch_funcs(df: pl.DataFrame) -> dict[str, list[str]]:
    cols = ["arch", "func"]
    out = {}
    for arch, func in df.select(cols).unique().sort(by=cols).iter_rows():
        out.setdefault(arch, []).append(func)
    return out


def make_func_wt(arch_funcs: dict[str, list[str]]) -> GroupedMultiSelect:
    return GroupedMultiSelect(
        arch_funcs, default_filter=lambda cat, s: s.endswith("Old") or s.endswith("New")
    )


def make_packed_width_one_pair_wt(unpacked_type_wt: MultiSelect, df: pl.DataFrame):
    unpacked_dict = build_type_sizes(df)
    packed_width_one_wt = widgets.BoundedIntText(
        value=1,
        min=0,
        max=max(unpacked_dict[t] for t in unpacked_type_wt.value),
        step=1,
        layout={"width": "60px"},
    )
    unpacked_dict = build_type_sizes(df)
    packed_width_one_slider_wt = widgets.IntSlider(
        min=0,
        max=packed_width_one_wt.max,
        value=packed_width_one_wt.value,
        step=1,
        orientation="horizontal",
        readout=False,
    )
    widgets.jslink(
        (packed_width_one_wt, "value"), (packed_width_one_slider_wt, "value")
    )
    widgets.jslink((packed_width_one_wt, "max"), (packed_width_one_slider_wt, "max"))

    def _set_max(val):
        packed_width_one_wt.max = max(unpacked_dict[t] for t in unpacked_type_wt.value)

    unpacked_type_wt.observe(_set_max, names="value")

    return packed_width_one_wt, packed_width_one_slider_wt


def build_palette(arch_funcs: dict[str, list[str]]):
    all_funcs = set()
    for fs in arch_funcs.values():
        all_funcs.update(fs)
    palette = sns.color_palette(
        f"tab{10 if len(all_funcs) <= 10 else 20}", len(all_funcs)
    )
    return dict(zip(sorted(all_funcs), palette))


def build_dashes(arch_funcs: dict[str, list[str]]):
    dash_styles = [(1, 0), (5, 5), (2, 2), (5, 2, 2, 2), (3, 1, 1, 1, 1, 1)]
    arches = sorted([a for a in arch_funcs.keys() if a != "Dynamic"])
    arches.append("Dynamic")  # Put last
    return dict(zip(arches, dash_styles))


def arch_func_filter(arch_funcs: dict[str, list[str]]):
    out = pl.lit(False)
    for arch, funcs in arch_funcs.items():
        out = out | ((pl.col("arch") == arch) & pl.col("func").is_in(funcs))
    return out


def plot(
    df: pl.DataFrame,
    unpacked_types: str,
    packed_width: int,
    arch_funcs: dict[str, list[str]],
    x_axis: tuple[int, int],
    out,
    **kwargs,
):
    selected_funcs = [f for fs in arch_funcs.values() for f in fs]
    data = (
        df.lazy()
        .filter(pl.col("unpacked_type").is_in(unpacked_types))
        .filter(pl.col("packed_width") == packed_width)
        .filter(arch_func_filter(arch_funcs))
        .filter(
            (x_axis[0] <= pl.col("num_values")) & (pl.col("num_values") <= x_axis[1])
        )
        .collect()
        .to_pandas()
    )
    all_funcs = df["func"].unique()  # Quick because categorical
    splits = {
        "hue": "func" if len(selected_funcs) > 1 else None,
        "col": "unpacked_type" if len(unpacked_types) > 1 else None,
        "style": "arch",
    }
    with out:
        out.clear_output(wait=True)
        g = sns.relplot(
            kind="line",
            data=data,
            x="num_values",
            y="cpu_time",
            **splits,
            **kwargs,
        )
        g.figure.subplots_adjust(top=0.9)
        g.fig.suptitle(
            "Unpack performance{func}{type}".format(
                func="" if len(selected_funcs) != 1 else f" for {funcs[0]} function",
                type="" if len(unpacked_types) != 1 else f" on {unpacked_types[0]}",
            )
        )
        for ax in g.axes.flat:
            ax.set_ylabel(f"CPU time (ns)")
            ax.set_xlabel(f"Integers to unpack (count)")
        plt.show()


def make_x_axis_wt(df: pl.DataFrame) -> widgets.IntRangeSlider:
    x_min, x_max = df.select(
        [
            pl.col("num_values").min().alias("min"),
            pl.col("num_values").max().alias("max"),
        ]
    ).row(0)
    return widgets.IntRangeSlider(
        value=[x_min, x_max],
        min=x_min,
        max=x_max,
        step=10,
        description="Horizontal axis:",
        style={"description_width": "initial"},
        orientation="horizontal",
        readout=True,
        readout_format="d",
    )


def make_results_explorer(df: pl.DataFrame):
    arch_funcs = make_arch_funcs(df)
    palette = build_palette(arch_funcs)
    dashes = build_dashes(arch_funcs)

    unpacked_type_wt = make_unpacked_type_wt(df=df)
    func_wt = make_func_wt(arch_funcs)
    packed_width_one_wt, packed_width_one_slider_wt = make_packed_width_one_pair_wt(
        unpacked_type_wt, df=df
    )
    x_axis_wt = make_x_axis_wt(df=df)
    out = widgets.Output()

    def plot_callback(*args, **kwargs):
        plot(
            df=df,
            unpacked_types=unpacked_type_wt.value,
            packed_width=packed_width_one_wt.value,
            arch_funcs=func_wt.value,
            x_axis=x_axis_wt.value,
            out=out,
            palette=palette,
            dashes=dashes,
            aspect=1.5,
            facet_kws={"sharex": False},
        )

    x_axis_wt.observe(plot_callback, names="value")
    unpacked_type_wt.observe(plot_callback, names="value")
    packed_width_one_wt.observe(plot_callback, names="value")
    func_wt.observe(plot_callback, names="value")

    controls = widgets.Tab(
        children=[
            # Functions
            widgets.VBox(
                [
                    widgets.Label("Select all function to compare:"),
                    func_wt,
                ]
            ),
            # Unpacked width
            widgets.VBox(
                [
                    widgets.Label("Select the target type for unpacking:"),
                    unpacked_type_wt,
                ],
                layout=widgets.Layout(min_height="100px"),
            ),
            # Packed width
            widgets.VBox(
                [
                    widgets.Label("Packed Width (in bits):"),
                    widgets.HBox([packed_width_one_slider_wt, packed_width_one_wt]),
                ]
            ),
            x_axis_wt,
        ],
        titles=["Functions", "Unpacked Type", "Packed Width", "View"],
        layout=widgets.Layout(min_height="200px"),
    )

    return widgets.VBox([controls, out]), plot_callback

In [ ]:
arch_wt = widgets.ToggleButtons(
    options=["linux-64", "osx-arm64"],
)
controls = widgets.VBox([])


def arch_callback(*args, **kwargs):
    df = load_cleaned_benchmarks(arch_wt.value)
    wt, start = make_results_explorer(df)
    controls.children = [arch_wt, wt]
    start()


arch_wt.observe(arch_callback, names="value")
arch_callback()

controls

### Speed agregation

In [ ]:
import numpy.polynomial as ply


def series_linear_regression(series, *args, **kwargs):
    x, y = series
    b, m = ply.Polynomial.fit(x.to_numpy(), y.to_numpy(), deg=1).convert().coef
    return pl.Series([{"slope": m, "intercept": b}])


def linear_regression(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df.group_by(["unpacked_type", "arch", "func", "packed_width", "unpacked_width"])
        .agg(
            pl.map_groups(
                exprs=["num_values", "cpu_time"],
                function=series_linear_regression,
                return_dtype=pl.Struct({"slope": pl.Float64, "intercept": pl.Float64}),
                returns_scalar=True,
            ).alias("fit")
        )
        .unnest("fit")
        .with_columns(
            (10**9 / pl.col("slope")).alias("values_per_second"),
            pl.col("intercept").alias("latency_ns"),
        )
        .drop("slope", "intercept")
    )


def plot_speed(
    df: pl.DataFrame,
    unpacked_types: str,
    arch_funcs: dict[str, list[str]],
    out,
    **kwargs,
):
    selected_funcs = [f for fs in arch_funcs.values() for f in fs]
    data = (
        df.lazy()
        .filter(pl.col("unpacked_type").is_in(unpacked_types))
        .filter(arch_func_filter(arch_funcs))
        .collect()
        .to_pandas()
    )
    all_funcs = df["func"].unique()  # Quick because categorical
    splits = {
        "hue": "func" if len(selected_funcs) > 1 else None,
        "col": "unpacked_type" if len(unpacked_types) > 1 else None,
        "style": "arch",
    }
    with out:
        out.clear_output(wait=True)
        g = sns.relplot(
            kind="line",
            data=data,
            x="packed_width",
            y="values_per_second",
            **splits,
            **kwargs,
        )
        g.figure.subplots_adjust(top=0.9)
        g.fig.suptitle(
            "Unpack speed {func}{type}".format(
                func="" if len(selected_funcs) != 1 else f" for {funcs[0]} function",
                type="" if len(unpacked_types) != 1 else f" on {unpacked_types[0]}",
            )
        )
        for ax in g.axes.flat:
            scale_10 = 9
            ax.yaxis.set_major_formatter(lambda x, _: f"{x / 10**scale_10:.1f}")
            ax.set_ylabel(f"speed (×10{'⁰¹²³⁴⁵⁶⁷⁸⁹'[scale_10]} int/s)")
            ax.set_xlabel(f"Packed width (bit)")
        plt.show()


def make_speed_results_explorer(df: pl.DataFrame):
    arch_funcs = make_arch_funcs(df)
    palette = build_palette(arch_funcs)
    dashes = build_dashes(arch_funcs)

    unpacked_type_wt = make_unpacked_type_wt(df=df)
    func_wt = make_func_wt(arch_funcs)
    out = widgets.Output()

    def plot_callback(*args, **kwargs):
        plot_speed(
            df=df,
            unpacked_types=unpacked_type_wt.value,
            arch_funcs=func_wt.value,
            out=out,
            palette=palette,
            dashes=dashes,
            aspect=1.5,
            facet_kws={"sharex": False},
        )

    unpacked_type_wt.observe(plot_callback, names="value")
    func_wt.observe(plot_callback, names="value")

    controls = widgets.Tab(
        children=[
            # Functions
            widgets.VBox(
                [
                    widgets.Label("Select all function to compare:"),
                    func_wt,
                ]
            ),
            # Unpacked width
            widgets.VBox(
                [
                    widgets.Label("Select the target type for unpacking:"),
                    unpacked_type_wt,
                ],
                layout=widgets.Layout(min_height="100px"),
            ),
        ],
        titles=["Functions", "Unpacked Type"],
        layout=widgets.Layout(min_height="200px"),
    )

    return widgets.VBox([controls, out]), plot_callback

In [ ]:
arch_wt = widgets.ToggleButtons(options=["linux-64", "osx-arm64"])
controls = widgets.VBox([])


def arch_callback(*args, **kwargs):
    df = load_cleaned_benchmarks(arch_wt.value)
    wt, start = make_speed_results_explorer(df=linear_regression(df))
    controls.children = [arch_wt, wt]
    start()


arch_wt.observe(arch_callback, names="value")
arch_callback()

controls